In [ ]:
# ================== MAPA DESDE MATRIZ PRECALCULADA ==================
import pandas as pd
import folium
from folium.features import CustomIcon, DivIcon
from folium.plugins import MarkerCluster
from shapely.geometry import MultiPoint, Point
import os
import glob
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

# Buscar archivos Excel en la carpeta de inputs
input_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Inputs_mapa'
archivos_excel = glob.glob(os.path.join(input_dir, '*.xlsx')) + glob.glob(os.path.join(input_dir, '*.xlsm'))

if archivos_excel:
    archivo_matriz = archivos_excel[0]  # Tomar el primer archivo encontrado
    print(f"Archivo seleccionado: {os.path.basename(archivo_matriz)}")
else:
    print("No se encontraron archivos Excel en la carpeta de inputs")
    archivo_matriz = None

output_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs'
os.makedirs(output_dir, exist_ok=True)
icon_met_path = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\95_24.png'
fecha_hora = datetime.now().strftime('%Y%m%d_%H%M%S')

# Leer la hoja de recorridos
df_recorridos = pd.read_excel(archivo_matriz, sheet_name='Recorridos')

# Usar los nombres reales de las columnas
col_met = '🏠 MET'
col_ruta = '🛣️ Ruta'
col_secuencia = '🔢 Secuencia'
col_tipo = 'Tipo'
col_codigo = '🆔 Codigo'
col_longitud = '🌍 Longitud'
col_latitud = '🌍 Latitud'
col_nombre = '📚 Nombre'
col_distancia_sig = '📏 Distancia_al_siguiente_km'

mets = sorted(df_recorridos[col_met].dropna().unique())
rutas = sorted(df_recorridos[col_ruta].dropna().unique())

# Selector interactivo de METs y rutas (dinámico)
met_selector = widgets.SelectMultiple(options=mets, value=tuple(mets), description='Selecciona MET(s):', style={'description_width': 'initial'}, layout=widgets.Layout(width='50%'))
ruta_selector = widgets.SelectMultiple(options=rutas, value=tuple(rutas), description='Selecciona Ruta(s):', style={'description_width': 'initial'}, layout=widgets.Layout(width='50%'))
todos_clientes_checkbox = widgets.Checkbox(value=True, description='Procesar todos los clientes', indent=False)
clientes_a_procesar = widgets.IntText(value=10, description='Cantidad de clientes a procesar:', style={'description_width': 'initial'}, layout=widgets.Layout(width='40%'))
generar_btn = widgets.Button(description='Generar mapa', button_style='success')
output_map = widgets.Output()

# Actualiza rutas disponibles según MET seleccionado
def actualizar_rutas(*args):
    mets_seleccionados = list(met_selector.value)
    if mets_seleccionados:
        rutas_filtradas = sorted(df_recorridos[df_recorridos[col_met].isin(mets_seleccionados)][col_ruta].dropna().unique())
    else:
        rutas_filtradas = sorted(df_recorridos[col_ruta].dropna().unique())
    ruta_selector.options = rutas_filtradas
    rutas_validas = [r for r in ruta_selector.value if r in rutas_filtradas]
    if rutas_validas:
        ruta_selector.value = tuple(rutas_validas)
    elif rutas_filtradas:
        ruta_selector.value = (rutas_filtradas[0],)
    else:
        ruta_selector.value = ()

met_selector.observe(actualizar_rutas, names='value')
actualizar_rutas()

# Paleta de colores para MET General
colores_met_general = ['#FF6B00', '#00529B', '#FFA500', '#8BC34A', '#E91E63', '#00BCD4', '#FF9800', '#9C27B0', '#607D8B', '#CDDC39', '#795548', '#3F51B5', '#F44336', '#009688', '#C0CA33', '#7B1FA2', '#D32F2F', '#388E3C', '#1976D2', '#FFA07A']

def generar_mapa(b):
    with output_map:
        clear_output()
        mets_seleccionados = list(met_selector.value)
        rutas_seleccionadas = list(ruta_selector.value)
        if not mets_seleccionados:
            print('Selecciona al menos un MET para generar el mapa.')
            return
        if not rutas_seleccionadas:
            print('Selecciona al menos una ruta para generar el mapa.')
            return

        met_row = df_recorridos[df_recorridos[col_met] == mets_seleccionados[0]].iloc[0]
        mapa = folium.Map(location=[met_row[col_latitud], met_row[col_longitud]], zoom_start=10, tiles='OpenStreetMap')

        featuregroups_combo = {}

        colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']

        resumen_combos = {}
        resumen_mets = {}
        color_idx = 0
        met_general_idx = 0
        met_general_ids = []
        # Crear una capa por cada combinación MET-Ruta
        for met in mets_seleccionados:
            met_clean = str(met).strip()
            if met_clean.startswith('MET '):
                met_clean = met_clean[4:]
            met_id = met_clean.replace(' ', '').replace(':', '').replace('.', '').replace('é', 'e').replace('á', 'a').replace('í', 'i').replace('ó', 'o').replace('ú', 'u')
            met_recorridos = df_recorridos[(df_recorridos[col_met] == met) & (df_recorridos[col_ruta].isin(rutas_seleccionadas))]
            rutas_met = met_recorridos[col_ruta].unique()
            for ruta in rutas_met:
                ruta_clean = str(ruta).strip()
                ruta_id = ruta_clean.replace(' ', '').replace(':', '').replace('.', '').replace('é', 'e').replace('á', 'a').replace('í', 'i').replace('ó', 'o').replace('ú', 'u')
                combo_id = f"{met_id}-{ruta_id}"
                capa_nombre = f"MET {met_clean} - Ruta {ruta_clean}"
                fg_combo = folium.FeatureGroup(name=capa_nombre, show=True)
                featuregroups_combo[combo_id] = fg_combo
                ruta_recorridos = met_recorridos[met_recorridos[col_ruta] == ruta]
                if ruta_recorridos.empty:
                    continue
                ruta_recorridos_sorted = ruta_recorridos.sort_values(col_secuencia)
                coords = list(zip(ruta_recorridos_sorted[col_latitud], ruta_recorridos_sorted[col_longitud]))
                color_ruta = colores[color_idx % len(colores)]
                color_idx += 1
                folium.PolyLine(coords, color=color_ruta, weight=5, opacity=0.8, tooltip=capa_nombre,
                    **{'data-met': met_id, 'data-ruta': ruta_id, 'data-combo': combo_id}
                ).add_to(fg_combo)
                for i, row in ruta_recorridos_sorted.iterrows():
                    popup_html = f'''
                    <div style=\"background: #fff; border-radius: 16px; box-shadow: 0 2px 8px rgba(0,0,0,0.15); padding: 14px; border: 2px solid #8BC34A; min-width: 260px; max-width: 340px;\">
                        <div style=\"font-size:20px; color:{color_ruta}; font-weight:bold; margin-bottom:4px;\">
                            <span style=\"vertical-align:middle;\">👨‍💼</span> {row[col_tipo]}: <span style=\"color:{color_ruta};\">{row[col_codigo]}</span>
                        </div>
                        <div style=\"font-size:16px; color:#333; margin-bottom:4px;\">
                            <span style=\"vertical-align:middle;\">📚</span> <b>Nombre:</b> {row[col_nombre]}
                        </div>
                        <div style=\"font-size:15px; color:#2A81CB; margin-bottom:4px;\">
                            <span style=\"vertical-align:middle;\">🗺️</span> <b>Ruta:</b> {row[col_ruta]}
                        </div>
                        <div style=\"font-size:15px; color:#CB2A2A; margin-bottom:4px;\">
                            <span style=\"vertical-align:middle;\">↩️</span> <b>Distancia al siguiente:</b> {row[col_distancia_sig]} km
                        </div>
                        <div style=\"font-size:15px; color:#FFC107; margin-bottom:4px;\">
                            <span style=\"vertical-align:middle;\">🔢</span> <b>Número de secuencia:</b> {row[col_secuencia]}
                        </div>
                        <div style=\"font-size:14px; color:#333; margin-bottom:2px;\">MET: <b>{met}</b></div>
                    </div>
                    '''
                    folium.Marker(
                        location=[row[col_latitud], row[col_longitud]],
                        popup=folium.Popup(popup_html, max_width=340),
                        icon=folium.Icon(color=color_ruta, icon='info-sign'),
                        **{'data-met': met_id, 'data-ruta': ruta_id, 'data-combo': combo_id}
                    ).add_to(fg_combo)
                    folium.Marker(
                        location=[row[col_latitud], row[col_longitud]],
                        icon=DivIcon(
                            icon_size=(24,24),
                            icon_anchor=(12,12),
                            html=f'<div style="font-size:16px; color:white; background:{color_ruta}; border-radius:50%; width:24px; height:24px; text-align:center; line-height:24px; border:2px solid #fff;">{row[col_secuencia]}</div>'
                        ),
                    ).add_to(fg_combo)
                met_rows = ruta_recorridos[ruta_recorridos[col_tipo] == 'MET']
                if not met_rows.empty:
                    met_row = met_rows.iloc[0]
                    folium.Marker(location=[met_row[col_latitud], met_row[col_longitud]],
                                  popup=folium.Popup(f'''<div style="background:#2A81CB; color:#fff; border-radius:14px; box-shadow:0 2px 8px rgba(0,0,0,0.15); padding:14px; border:2px solid #fff; min-width:220px; text-align:center;">
                                    <div style="font-size:20px; font-weight:bold; margin-bottom:6px;">🏠 MET: <span style="color:#FFD600;">{met}</span></div>
                                    <div style="font-size:15px; color:#fff; margin-bottom:2px;">Coordenadas:<br><span style="font-size:14px; color:#FFD600;">{met_row[col_latitud]}, {met_row[col_longitud]}</span></div>
                                  </div>''', max_width=260),
                                  icon=CustomIcon(icon_met_path, icon_size=(32,32))).add_to(fg_combo)

                fg_combo.add_to(mapa)

                clientes_ruta = ruta_recorridos[ruta_recorridos[col_tipo] == 'Cliente']
                total_clientes_ruta = clientes_ruta.shape[0]
                distancia_total_ruta = ruta_recorridos['📏 Distancia_total_ruta_km'].dropna().unique()
                distancia_total_ruta = distancia_total_ruta[0] if len(distancia_total_ruta) > 0 else 0
                distancia_promedio_ruta = distancia_total_ruta / total_clientes_ruta if total_clientes_ruta > 0 else 0
                resumen_combos[combo_id] = {
                    'met': met,
                    'ruta': ruta,
                    'clientes': total_clientes_ruta,
                    'distancia_total_km': distancia_total_ruta,
                    'distancia_promedio_cliente_km': distancia_promedio_ruta,
                    'color': color_ruta
                }
            # Crear una capa general por MET (solo polígono envolvente y cluster de clientes)
            fg_met = folium.FeatureGroup(name=f"MET {met_clean} (General)", show=False)
            met_general_ids.append(f"MET {met_clean} (General)")
            color_met_general = colores_met_general[met_general_idx % len(colores_met_general)]
            met_general_idx += 1
            clientes_met = met_recorridos[met_recorridos[col_tipo] == 'Cliente']
            puntos = [Point(row[col_longitud], row[col_latitud]) for _, row in clientes_met.iterrows()]
            if len(puntos) > 2:
                multipoint = MultiPoint(puntos)
                hull = multipoint.convex_hull
                if hull.geom_type == 'Polygon':
                    coords = [(lat, lon) for lon, lat in hull.exterior.coords]
                    popup_met = f'''
                    <div style="width:210px; padding:9px 10px; background:{color_met_general}; color:#fff; border-radius:7px; box-shadow:0 1px 6px #aaa; font-size:11px; font-weight:bold; text-align:center;">
                        <span style='font-size:13px;'>🏠 MET: {met_clean} (General)</span>
                    </div>'''
                    folium.Polygon(locations=coords, color=color_met_general, fill=True, fill_opacity=0.25, popup=folium.Popup(popup_met, max_width=225)).add_to(fg_met)
            # Cluster de clientes
            cluster = MarkerCluster(name=f'Clientes MET {met_clean} (General)')
            for _, row in clientes_met.iterrows():
                popup_html = f'''
                <div style="background: #fff; border-radius: 14px; box-shadow: 0 2px 8px rgba(0,0,0,0.15); padding: 12px; border: 2px solid #8BC34A; min-width: 220px; max-width: 320px;">
                    <div style="font-size:18px; color:gray; font-weight:bold; margin-bottom:4px;">
                        <span style="vertical-align:middle;">👨‍💼</span> Cliente: <span style="color:gray;">{row[col_codigo]}</span>
                    </div>
                    <div style="font-size:15px; color:#333; margin-bottom:4px;">
                        <span style="vertical-align:middle;">📚</span> <b>Nombre:</b> {row[col_nombre]}
                    </div>
                    <div style="font-size:14px; color:#2A81CB; margin-bottom:4px;">
                        <span style="vertical-align:middle;">🗺️</span> <b>Ruta:</b> {row[col_ruta]}
                    </div>
                </div>'''
                folium.Marker(
                    location=[row[col_latitud], row[col_longitud]],
                    popup=folium.Popup(popup_html, max_width=320),
                    icon=folium.Icon(color='gray', icon='user'),
                ).add_to(cluster)
            cluster.add_to(fg_met)
            # Marcador MET principal
            met_rows = met_recorridos[met_recorridos[col_tipo] == 'MET']
            if not met_rows.empty:
                met_row = met_rows.iloc[0]
                folium.Marker(location=[met_row[col_latitud], met_row[col_longitud]],
                              popup=folium.Popup(f'''<div style="background:#2A81CB; color:#fff; border-radius:14px; box-shadow:0 2px 8px rgba(0,0,0,0.15); padding:14px; border:2px solid #fff; min-width:220px; text-align:center;">
                                <div style="font-size:20px; font-weight:bold; margin-bottom:6px;">🏠 MET: <span style="color:#FFD600;">{met_clean}</span></div>
                                <div style="font-size:15px; color:#fff; margin-bottom:2px;">Coordenadas:<br><span style="font-size:14px; color:#FFD600;">{met_row[col_latitud]}, {met_row[col_longitud]}</span></div>
                              </div>''', max_width=260),
                              icon=CustomIcon(icon_met_path, icon_size=(32,32))).add_to(fg_met)
            fg_met.add_to(mapa)
            # Resumen general por MET
            total_clientes_met = clientes_met.shape[0]
            distancia_total_met = met_recorridos['📏 Distancia_total_ruta_km'].dropna().sum()
            distancia_promedio_met = distancia_total_met / total_clientes_met if total_clientes_met > 0 else 0
            resumen_mets[met_id] = {
                'met': met_clean,
                'clientes': total_clientes_met,
                'distancia_total_km': distancia_total_met,
                'distancia_promedio_cliente_km': distancia_promedio_met,
                'color': 'gray'
            }

        folium.LayerControl(collapsed=False, autoZIndex=True).add_to(mapa)

        # CSS dentro de <style> para scroll en el control de capas
        scroll_layers_css = '''<style>.leaflet-control-layers-list { max-height: 220px; overflow-y: auto; overflow-x: hidden; width: 100%; min-width: 180px; }</style>'''
        mapa.get_root().html.add_child(folium.Element(scroll_layers_css))

        titulo_html = f'''<div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background-color: white; padding: 7px 20px; border: 2px solid #333; border-radius: 12px; box-shadow: 0 2px 8px rgba(0,0,0,0.3);"><h2 style="margin: 0; color: #333; text-align: center; font-size:14px;"><span style='font-size:14px;'>🗺️ MAPA RECORRIDOS ÓPTIMOS METS-CLIENTE (Matriz Precalculada)</span></h2><p style="margin: 4px 0 0 0; text-align: center; color: #666; font-size: 9px;">METS analizados: <b>{len(mets_seleccionados)}</b> | Rutas: <b>{len(rutas_seleccionadas)}</b></p></div>'''
        mapa.get_root().html.add_child(folium.Element(titulo_html))

        resumen_html = (
            f'<div id="resumen-general" style="position: fixed; top: 20px; right: 35px; width: 340px; background-color: white; border: 2px solid #333; z-index: 9997; font-size: 12px; padding: 14px; border-radius: 12px; box-shadow: 0 2px 8px rgba(0,0,0,0.3);">'
        )
        resumen_html += "<div id='mensaje-resumen' style='color:#888; font-size:13px; text-align:center; margin-bottom:8px;'>Activa una capa para ver el resumen.</div>"
        # Generar resumen para cada combinación MET-Ruta, aunque no haya datos
        for met in mets_seleccionados:
            met_clean = str(met).strip()
            if met_clean.startswith('MET '):
                met_clean = met_clean[4:]
            met_id = met_clean.replace(' ', '').replace(':', '').replace('.', '').replace('é', 'e').replace('á', 'a').replace('í', 'i').replace('ó', 'o').replace('ú', 'u')
            met_recorridos = df_recorridos[(df_recorridos[col_met] == met) & (df_recorridos[col_ruta].isin(rutas_seleccionadas))]
            rutas_met = met_recorridos[col_ruta].unique()
            for ruta in rutas_met:
                ruta_clean = str(ruta).strip()
                ruta_id = ruta_clean.replace(' ', '').replace(':', '').replace('.', '').replace('é', 'e').replace('á', 'a').replace('í', 'i').replace('ó', 'o').replace('ú', 'u')
                combo_id = f"{met_id}-{ruta_id}"
                resumen = resumen_combos.get(combo_id)
                if resumen:
                    resumen_html += (
                        f'<div id="resumen-{combo_id}" class="resumen-combo" style="display:block; margin-bottom:10px;">'
                        f'<h3 style="margin-top: 0; color: {resumen["color"]}; text-align: left; border-bottom: 2px solid #ddd; padding-bottom: 5px; font-size:14px;"><span style="font-size:14px;">🏠 MET {met_clean} - 🛣️ Ruta {ruta_clean}</span></h3>'
                        f'<p style="margin: 5px 0; font-weight: bold; color: #333; font-size:12px;">🧮 Total clientes: {resumen["clientes"]}</p>'
                        f'<p style="margin: 6px 0 4px 0; color: #444; font-size:11px;">🚗 <b>Distancia total:</b> {resumen["distancia_total_km"]:.2f} km</p>'
                        f'<p style="margin: 4px 0 0 0; color: {resumen["color"]}; font-size:11px;">Promedio por cliente: {resumen["distancia_promedio_cliente_km"]:.2f} km</p>'
                        f'</div>'
                    )
                else:
                    resumen_html += (
                        f'<div id="resumen-{combo_id}" class="resumen-combo" style="display:block; margin-bottom:10px;">'
                        f'<h3 style="margin-top: 0; color: #CB2A2A; text-align: left; border-bottom: 2px solid #ddd; padding-bottom: 5px; font-size:14px;"><span style="font-size:14px;">🏠 MET {met_clean} - 🛣️ Ruta {ruta_clean}</span></h3>'
                        "<p style='color:#CB2A2A; font-size:12px;'>No hay datos para la selección actual.</p>"
                        f'</div>'
                    )
        # Resumen por MET (general)
        for met_id, resumen in resumen_mets.items():
            resumen_html += (
                f'<div id="resumen-met-{met_id}" class="resumen-met" style="display:block; margin-bottom:10px;">'
                f'<h3 style="margin-top: 0; color: {resumen["color"]}; text-align: left; border-bottom: 2px solid #ddd; padding-bottom: 5px; font-size:14px;"><span style="font-size:14px;">🏠 MET {resumen["met"]} (General)</span></h3>'
                f'<p style="margin: 5px 0; font-weight: bold; color: #333; font-size:12px;">🧮 Total clientes: {resumen["clientes"]}</p>'
                f'<p style="margin: 6px 0 4px 0; color: #444; font-size:11px;">🚗 <b>Distancia total:</b> {resumen["distancia_total_km"]:.2f} km</p>'
                f'<p style="margin: 4px 0 0 0; color: {resumen["color"]}; font-size:11px;">Promedio por cliente: {resumen["distancia_promedio_cliente_km"]:.2f} km</p>'
                f'</div>'
            )
        resumen_html += "</div>"
        mapa.get_root().html.add_child(folium.Element(resumen_html))

        # JS para mostrar/ocultar resumen y capas por combinación MET-Ruta
        update_resumen_js = r'''<script>
function normalizarId(valor) {
  // Quitar prefijo 'MET', espacios, dos puntos, puntos y acentos igual que en Python
  valor = valor.trim();
  if (valor.startsWith('MET ')) valor = valor.substring(4);
  valor = valor.replace(/\s/g, '').replace(/:/g, '').replace(/\./g, '');
  valor = valor.replace(/[éÉ]/g, 'e').replace(/[áÁ]/g, 'a').replace(/[íÍ]/g, 'i').replace(/[óÓ]/g, 'o').replace(/[úÚ]/g, 'u');
  return valor;
}
document.addEventListener('DOMContentLoaded', function() {
  function updateResumen() {
    var comboActivos = {};
    var overlays = document.querySelectorAll('.leaflet-control-layers-overlays label');
    overlays.forEach(function(label) {
      var input = label.querySelector('input');
      var span = label.querySelector('span:last-child');
      if (!span) return;
      var text = span.textContent.trim();
      if (text.startsWith('MET')) {
        // Extraer MET y Ruta
        var partes = text.match(/^MET\s(.+)\s-\sRuta\s(.+)$/);
        if (partes) {
          var met = partes[1];
          var ruta = partes[2];
          var comboId = normalizarId(met) + '-' + normalizarId(ruta);
          comboActivos[comboId] = input.checked;
        }
      }
    });
    var resumenComboDivs = document.querySelectorAll('.resumen-combo');
    var algunoActivo = false;
    resumenComboDivs.forEach(function(div) {
      var comboMatch = div.id.match(/resumen-(.+)/);
      if (comboMatch) {
        var combo = comboMatch[1];
        var visible = comboActivos[combo];
        div.style.display = visible ? 'block' : 'none';
        if (visible) algunoActivo = true;
      }
    });
    var mensajeResumen = document.getElementById('mensaje-resumen');
    if (!algunoActivo && mensajeResumen) {
      mensajeResumen.style.display = 'block';
    } else if (mensajeResumen) {
      mensajeResumen.style.display = 'none';
    }
    var resumenGeneral = document.getElementById('resumen-general');
    if (resumenGeneral) {
      resumenGeneral.style.display = 'block';
    }
    var polylines = document.querySelectorAll('.leaflet-interactive');
    polylines.forEach(function(line) {
      var combo = line.getAttribute('data-combo');
      if (combo) {
        line.style.display = comboActivos[combo] ? 'block' : 'none';
      }
    });
    var markers = document.querySelectorAll('.leaflet-marker-icon, .leaflet-div-icon');
    markers.forEach(function(marker) {
      var combo = marker.getAttribute('data-combo');
      if (combo) {
        marker.style.display = comboActivos[combo] ? 'block' : 'none';
      }
    });
  }
  var overlays = document.querySelectorAll('.leaflet-control-layers-overlays input');
  overlays.forEach(function(input) {
    input.addEventListener('change', updateResumen);
  });
  updateResumen();
});
</script>'''
        mapa.get_root().html.add_child(folium.Element(update_resumen_js))

        # JS para mover el control de capas debajo del control de zoom
        move_layers_js = r'''<script>
document.addEventListener('DOMContentLoaded', function() {
  var zoom = document.querySelector('.leaflet-control-zoom');
  var layers = document.querySelector('.leaflet-control-layers');
  if (zoom && layers) {
    zoom.parentNode.insertBefore(layers, zoom.nextSibling);
  }
});
</script>'''
        mapa.get_root().html.add_child(folium.Element(move_layers_js))

        # JS para seleccionar solo las capas MET General al abrir el mapa
        seleccionar_met_general_js = r'''<script>
document.addEventListener('DOMContentLoaded', function() {
  var labels = document.querySelectorAll('.leaflet-control-layers-overlays label');
  labels.forEach(function(label) {
    var input = label.querySelector('input');
    var span = label.querySelector('span:last-child');
    if (!span || !input) return;
    var text = span.textContent.trim();
    if (text.endsWith('(General)')) {
      input.checked = true;
    } else {
      input.checked = false;
    }
    var event = new Event('change');
    input.dispatchEvent(event);
  });
});
</script>'''
        mapa.get_root().html.add_child(folium.Element(seleccionar_met_general_js))

        # JS para ocultar visualmente las líneas y marcadores de capas no seleccionadas al abrir el mapa
        ocultar_no_general_js = r'''<script>
document.addEventListener('DOMContentLoaded', function() {
  var labels = document.querySelectorAll('.leaflet-control-layers-overlays label');
  var metGeneralNames = [];
  labels.forEach(function(label) {
    var span = label.querySelector('span:last-child');
    if (!span) return;
    var text = span.textContent.trim();
    if (text.endsWith('(General)')) {
      metGeneralNames.push(text);
    }
  });
  // Oculta polylines y marcadores de combos no seleccionados
  var overlays = document.querySelectorAll('.leaflet-control-layers-overlays label');
  overlays.forEach(function(label) {
    var input = label.querySelector('input');
    var span = label.querySelector('span:last-child');
    if (!span || !input) return;
    var text = span.textContent.trim();
    if (!text.endsWith('(General)')) {
      input.checked = false;
      var event = new Event('change');
      input.dispatchEvent(event);
    } else {
      input.checked = true;
      var event = new Event('change');
      input.dispatchEvent(event);
    }
  });
  // Forzar ocultar visualmente polylines y marcadores de combos no seleccionados
  setTimeout(function() {
    var labels = document.querySelectorAll('.leaflet-control-layers-overlays label');
    labels.forEach(function(label) {
      var input = label.querySelector('input');
      var span = label.querySelector('span:last-child');
      if (!span || !input) return;
      var checked = input.checked;
      var comboId = null;
      if (text.startsWith('MET') && text.includes('- Ruta')) {
        var partes = text.match(/^MET\s(.+)\s-\sRuta\s(.+)$/);
        if (partes) {
          var met = partes[1];
          var ruta = partes[2];
          comboId = met.replace(/\s/g, '').replace(/:/g, '').replace(/\./g, '').replace(/[éÉ]/g, 'e').replace(/[áÁ]/g, 'a').replace(/[íÍ]/g, 'i').replace(/[óÓ]/g, 'o').replace(/[úÚ]/g, 'u') + '-' + ruta.replace(/\s/g, '').replace(/:/g, '').replace(/\./g, '').replace(/[éÉ]/g, 'e').replace(/[áÁ]/g, 'a').replace(/[íÍ]/g, 'i').replace(/[óÓ]/g, 'o').replace(/[úÚ]/g, 'u');
        }
      }
      // Oculta polylines y marcadores de combos no seleccionados
      if (comboId && !checked) {
        var polylines = document.querySelectorAll('.leaflet-interactive[data-combo="' + comboId + '"]');
        polylines.forEach(function(line) { line.style.display = 'none'; });
        var markers = document.querySelectorAll('.leaflet-marker-icon[data-combo="' + comboId + '"], .leaflet-div-icon[data-combo="' + comboId + '"]');
        markers.forEach(function(marker) { marker.style.display = 'none'; });
      }
    });
  }, 500);
});
</script>'''
        mapa.get_root().html.add_child(folium.Element(ocultar_no_general_js))

        # JS para deseleccionar todas las capas al abrir el mapa
        deseleccionar_todas_js = r'''<script>
document.addEventListener('DOMContentLoaded', function() {
  var labels = document.querySelectorAll('.leaflet-control-layers-overlays label');
  labels.forEach(function(label) {
    var input = label.querySelector('input');
    if (!input) return;
    input.checked = false;
    var event = new Event('change');
    input.dispatchEvent(event);
  });
  // Forzar ocultar visualmente polylines y marcadores de combos no seleccionados
  setTimeout(function() {
    var labels = document.querySelectorAll('.leaflet-control-layers-overlays label');
    labels.forEach(function(label) {
      var input = label.querySelector('input');
      var span = label.querySelector('span:last-child');
      if (!span || !input) return;
      var checked = input.checked;
      var comboId = null;
      var text = span.textContent.trim();
      if (text.startsWith('MET') && text.includes('- Ruta')) {
        var partes = text.match(/^MET\s(.+)\s-\sRuta\s(.+)$/);
        if (partes) {
          var met = partes[1];
          var ruta = partes[2];
          comboId = met.replace(/\s/g, '').replace(/:/g, '').replace(/\./g, '').replace(/[éÉ]/g, 'e').replace(/[áÁ]/g, 'a').replace(/[íÍ]/g, 'i').replace(/[óÓ]/g, 'o').replace(/[úÚ]/g, 'u') + '-' + ruta.replace(/\s/g, '').replace(/:/g, '').replace(/\./g, '').replace(/[éÉ]/g, 'e').replace(/[áÁ]/g, 'a').replace(/[íÍ]/g, 'i').replace(/[óÓ]/g, 'o').replace(/[úÚ]/g, 'u');
        }
      }
      // Oculta polylines y marcadores de combos no seleccionados
      if (comboId && !checked) {
        var polylines = document.querySelectorAll('.leaflet-interactive[data-combo="' + comboId + '"]');
        polylines.forEach(function(line) { line.style.display = 'none'; });
        var markers = document.querySelectorAll('.leaflet-marker-icon[data-combo="' + comboId + '"], .leaflet-div-icon[data-combo="' + comboId + '"]');
        markers.forEach(function(marker) { marker.style.display = 'none'; });
      }
      // Oculta polígono y cluster de MET General si no está checked
      if (text.endsWith('(General)') && !checked) {
        var fgNames = document.querySelectorAll('.leaflet-control-layers-overlays span');
        fgNames.forEach(function(spanName) {
          if (spanName.textContent.trim() === text) {
            var parent = spanName.closest('.leaflet-control-layers-overlays label');
            if (parent) {
              // No hay data-combo para general, pero se oculta por checked
              // El control de folium ya lo hace, pero se fuerza aquí
            }
          }
        });
      }
    });
  }, 500);
});
</script>'''
        mapa.get_root().html.add_child(folium.Element(deseleccionar_todas_js))

        nombre_mapa = os.path.join(output_dir, f'mapa_recorridos_matriz_precalculada_{fecha_hora}.html')
        try:
            mapa.save(nombre_mapa)
            print(f'Mapa grupal exportado a: {nombre_mapa}')
        except Exception as e:
            print(f'Error al guardar el mapa: {e}')

generar_btn.on_click(generar_mapa)
display(widgets.VBox([
    widgets.HTML('<b>Control de capas jerárquico y dinámico por combinación MET-Ruta:</b> Selecciona primero el MET y luego la ruta. El selector de rutas se actualiza automáticamente.'),
    met_selector,
    ruta_selector,
    widgets.HBox([clientes_a_procesar, todos_clientes_checkbox]),
    generar_btn,
    output_map
]))

Archivo seleccionado: recorridos_secuencia_todos_20260213_060416.xlsx
